In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
import os
import sys
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, make_scorer
import lightgbm as lgb
import optuna
# append source to sys path
src_path = '../src' 
if src_path not in sys.path:
    sys.path.append(src_path)

import dataprocessing as dp
import model 


/home/william/.pyenv/versions/3.13.1/envs/solarpred/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
A = dp.read_data('A', reindex=False)
B = dp.read_data('B', reindex=False)
C = dp.read_data('C', reindex=False)

raw_data = pd.concat([A, B, C], ignore_index=True)
data = model.preprocess_data(raw_data)
data


Redundant feature removal (Pearson correlation >= 0.93):
  Original features: 48
  Redundant pairs found: 11
  Redundant pairs: [('clear_sky_rad:W', 'clear_sky_energy_1h:J'), ('dew_point_2m:K', 'absolute_humidity_2m:gm3'), ('diffuse_rad_1h:J', 'diffuse_rad:W'), ('direct_rad_1h:J', 'direct_rad:W'), ('pressure_100m:hPa', 'msl_pressure:hPa'), ('pressure_50m:hPa', 'msl_pressure:hPa'), ('pressure_50m:hPa', 'pressure_100m:hPa'), ('sfc_pressure:hPa', 'msl_pressure:hPa'), ('sfc_pressure:hPa', 'pressure_100m:hPa'), ('sfc_pressure:hPa', 'pressure_50m:hPa'), ('total_cloud_cover:p', 'effective_cloud_cover:p')]
  Features to drop: ['absolute_humidity_2m:gm3', 'clear_sky_energy_1h:J', 'diffuse_rad_1h:J', 'direct_rad_1h:J', 'msl_pressure:hPa', 'pressure_50m:hPa', 'sfc_pressure:hPa', 'total_cloud_cover:p']
  Features remaining: 40
Total features to drop: {'snow_drift:idx', 'is_in_shadow:idx', 'dew_point_2m:K', 't_1000hPa:K', 'wind_speed_v_10m:ms', 'pressure_100m:hPa', 'fresh_snow_24h:cm', 'sun_elevat

,ceiling_height_agl:m,clear_sky_rad:W,cloud_base_agl:m,dew_or_rime:idx,diffuse_rad:W,direct_rad:W,effective_cloud_cover:p,fresh_snow_1h:cm,fresh_snow_3h:cm,fresh_snow_6h:cm,...,wind_speed_u_10m:ms,wind_speed_w_1000hPa:ms,building,pv_measurement,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos
0,1744.900024,0.0,1744.900024,0.0,0.0,0.0,98.699997,0.0,0.0,0.0,...,-3.6,-0.0,0,0.00,-0.500000,0.866025,0.486273,-0.873807,1.224647e-16,-1.0
1,1703.599976,0.0,1703.599976,0.0,0.0,0.0,99.599998,0.0,0.0,0.0,...,-3.5,-0.0,0,0.00,-0.258819,0.965926,0.486273,-0.873807,1.224647e-16,-1.0
2,1668.099976,0.0,1668.099976,0.0,0.0,0.0,100.000000,0.0,0.0,0.0,...,-3.1,-0.0,0,0.00,0.000000,1.000000,0.471160,-0.882048,1.224647e-16,-1.0
3,1388.400024,0.0,1388.400024,0.0,0.0,0.0,100.000000,0.0,0.0,0.0,...,-2.7,-0.0,0,0.00,0.258819,0.965926,0.471160,-0.882048,1.224647e-16,-1.0
4,1108.500000,9.8,1108.500000,0.0,4.3,0.0,100.000000,0.0,0.0,0.0,...,-2.5,-0.0,0,19.36,0.500000,0.866025,0.471160,-0.882048,1.224647e-16,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99009,1474.199951,13.4,557.000000,0.0,8.8,0.0,98.599998,0.0,0.0,0.0,...,3.9,-0.0,2,50.96,-0.965926,0.258819,0.880012,-0.474951,8.660254e-01,-0.5
99010,1427.300049,0.0,541.700012,0.0,0.0,0.0,97.400002,0.0,0.0,0.0,...,3.1,-0.0,2,2.94,-0.866025,0.500000,0.880012,-0.474951,8.660254e-01,-0.5
99011,1558.099976,0.0,601.500000,0.0,0.0,0.0,92.099998,0.0,0.0,0.0,...,2.7,-0.0,2,0.00,-0.707107,0.707107,0.880012,-0.474951,8.660254e-01,-0.5
99012,1446.599976,0.0,540.700012,0.0,0.0,0.0,96.500000,0.0,0.0,0.0,...,2.6,-0.0,2,-0.00,-0.500000,0.866025,0.880012,-0.474951,8.660254e-01,-0.5


In [ ]:
study_name = 'lgbm_pv_wide_optimization_cv'
last_study_name = 'lgbm_pv_optimization_narrowed4'
# try:
#     study = optuna.create_study(direction='minimize', study_name=study_name, storage='sqlite:///lgbm_pv_optimization.db')
# except:
#     optuna.delete_study(study_name=study_name, storage='sqlite:///lgbm_pv_optimization.db')
#     study = optuna.create_study(direction='minimize', study_name=study_name, storage='sqlite:///lgbm_pv_optimization.db')
study = optuna.create_study(
    direction='minimize',
    study_name=study_name,
    storage='sqlite:///lgbm_pv_optimization.db',
    load_if_exists=True,
)

data  = data.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
data.dropna(subset=['pv_measurement'], inplace=True)

X = data.drop(columns=['pv_measurement'])
y = data['pv_measurement']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42) # 0.25 x 0.8 = 0.2

default_params = {
    "objective": "regression_l1",
    'metric': 'mae',
    "random_state":0,
    "n_estimators":500, 
    "bagging_freq":1,
    'learning_rate':  0.06389602179214016,  
    'num_leaves': 248, 
    'min_data_in_leaf': 2, 
    'bagging_fraction': 0.838404028491540, 
    'colsample_bytree': 0.7284801370182925,
    'early_stopping_round':50
}

previous_study = optuna.load_study(study_name=last_study_name, storage='sqlite:///lgbm_pv_optimization.db')
previous_best_params = previous_study.best_trial.params

def objective(trial):
 

    # max_depth_choice = trial.suggest_categorical("max_depth_choice", [-1, "range"])
    
    # if max_depth_choice == "range":
    #     max_depth = trial.suggest_int("max_depth_in_range", 20, 40)
    # else:
    #     max_depth = -1  # Unlimited depth

    params = {
        "objective": "regression_l1",
        'metric': 'mae',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True), 
        'num_leaves': trial.suggest_int('num_leaves', 20, 600),
        'max_depth': trial.suggest_int('max_depth', -50, 50),
        # 'n_estimators': trial.suggest_int('n_estimators', 600, 900), 
        "n_estimators": 600,
        # "bagging_freq":trial.suggest_int('bagging_freq', 0, 1),
        "bagging_freq":trial.suggest_int('bagging_freq', 0,5),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 0, 100), 
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.100, 0.990),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.100, 0.98),
        'random_state': 42,
        # 'early_stopping_round':50,
        'verbose': -1
    }

    
    # train = lgb.Dataset(X_train, label=y_train)
    # valid = lgb.Dataset(X_val, label=y_val, reference=train)

    # model = lgb.train(params, train) 
    # preds = model.predict(X_val)
    # score = mean_squared_error(y_val, preds)

    model = lgb.LGBMRegressor(**params)

    #cross validate
    mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring=mae_scorer, n_jobs=-1)

    # model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
    # preds = model.predict(X_val)
    # score = mean_absolute_error(y_val, preds)
    
    return scores.mean() * -1  # Negate because greater_is_better=False for sklearn cross_val_score


previous_best_params.pop('early_stopping_round', None)  # Remove early_stopping_round if it exists (not used in cross-val)
default_params.pop('early_stopping_round', None)  

study.enqueue_trial(previous_best_params)
study.enqueue_trial(default_params)
study.enqueue_trial(default_params)
study.optimize(objective, n_trials=100, n_jobs=-1)
print("Best trial:")
trial = study.best_trial
print(f"  Value: {trial.value}")
# print(f"  Score: {trial.score}")
print(f" Params: {trial.params}")

model = lgb.LGBMRegressor(**trial.params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
preds = model.predict(X_test)
score = mean_absolute_error(y_test, preds)
print(f"Initial model test MSE: {score}")



[I 2025-10-31 21:23:22,123] Using an existing study with name 'lgbm_pv_wide_optimization_cv' instead of creating a new one.


In [ ]:
trial = study.best_trial
model = lgb.LGBMRegressor(**trial.params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)])
preds = model.predict(X_test)
score = mean_absolute_error(y_test, preds)
print(f"Initial model test MAE: {score}")

[LightGBM] [Warning] min_data_in_leaf is set=9, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=9
[LightGBM] [Warning] bagging_fraction is set=0.8717486209269667, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8717486209269667
[LightGBM] [Warning] min_data_in_leaf is set=9, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=9
[LightGBM] [Warning] bagging_fraction is set=0.8717486209269667, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8717486209269667
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009070 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3164
[LightGBM] [Info] Number of data points in the train set: 55770, number of used features: 29
[LightGBM] [Warning] min_data_in_leaf is set=9, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=9
[LightGBM] [Warning] bagging_fraction is set=0.8717486209

In [ ]:
X2_train, X2_val, y2_train, y2_val = train_test_split(data.drop(columns=['pv_measurement']), data['pv_measurement'].to_numpy(), test_size=0.2, random_state=42) 
print(X2_train.shape)
print(X2_val.shape)
default_params = {
        "objective": "regression_l1",
        'metric': 'mae',
        "random_state":0,
        "n_estimators":500, 
        "bagging_freq":1,
        'learning_rate':  0.06389602179214016,  
        'num_leaves': 248, 
        'min_data_in_leaf': 2, 
        'bagging_fraction': 0.838404028491540, 
        'colsample_bytree': 0.7284801370182925,
        # 'early_stopping_round':50
    }

model2 = lgb.LGBMRegressor(**default_params)
# model2.fit(X = X2_train, y = y2_train, eval_set=[(X2_val, y2_val)])
# preds2 = model2.predict(X2_val)
# score2 = mean_absolute_error(y2_val, preds2)

mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False) #MAE must be negated for cross_val_score
scores2 = cross_val_score(model2, X, y, cv=5, scoring=mae_scorer, n_jobs=2)
print(f"Default model test MAE: {scores2.mean()}")

model3 = lgb.LGBMRegressor(**trial.params)
# model3.fit(X = X2_train, y = y2_train, eval_set=[(X2_val, y2_val)])
# preds3 = model3.predict(X2_val)
# score3 = mean_absolute_error(y2_val, preds3)

scores3 = cross_val_score(model2, X, y, cv=5, scoring=mae_scorer, n_jobs=2)

print(f"Tuned model test MAE: {scores3.mean()}")

(74360, 29)
(18591, 29)
[LightGBM] [Warning] min_data_in_leaf is set=2, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=2
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_fraction is set=0.83840402849154, subsample=1.0 will be ignored. Current value: bagging_fraction=0.83840402849154
[LightGBM] [Warning] min_data_in_leaf is set=2, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=2
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_fraction is set=0.83840402849154, subsample=1.0 will be ignored. Current value: bagging_fraction=0.83840402849154
[LightGBM] [Warning] min_data_in_leaf is set=2, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=2
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[Light

AttributeError: 'float' object has no attribute 'mean'

In [ ]:
print(f"Default model test MAE: {scores2.mean()}")
print(f"Tuned model test MAE: {scores3.mean()}")

Default model test MAE: -94.26500570012405
Tuned model test MAE: -94.26500570012405


In [ ]:
print(f" Params: {trial.params}")
print(f"  Value: {trial.value}")


 Params: {'learning_rate': 0.04697839159113882, 'num_leaves': 342, 'max_depth': 36, 'min_data_in_leaf': 9, 'bagging_fraction': 0.8717486209269667, 'colsample_bytree': 0.9199547171951525}
  Value: 80.38270640859984


In [ ]:
params = trial.params
params['early_stopping_round'] = 50
# params = {'learning_rate': 0.08676161102092138, 'num_leaves': 85, 'n_estimators': 970, 'min_data_in_leaf': 5, 'bagging_fraction': 0.6830888152713105, 'colsample_bytree': 0.9874194827341676, 'early_stopping_round': 50}
lgbm_model, y_val, x_val = model.train_lightgbm_model(data, params=params) #prev 78.5464

AttributeError: 'LGBMRegressor' object has no attribute 'train_lightgbm_model'

In [ ]:
lgbm_model.eval_valid()

In [ ]:
lgbm_model.feature_importance(importance_type='gain')

importance_df = pd.DataFrame({
    'feature': lgbm_model.feature_name(),
    'importance_gain': lgbm_model.feature_importance(importance_type='gain'),
    'importance_split': lgbm_model.feature_importance(importance_type='split')
})#.sort_values(by='importance_split', ascending=False).reset_index(drop=True)

print(x_val.columns)
importance_df

In [ ]:
month = 6
sin_value = np.sin(2 * np.pi * month / 12)
cos_value = np.cos(2 * np.pi * month / 12)

print(f"sin(June): {sin_value}")
print(f"cos(June): {cos_value}")